In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

# Load data
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')

print("Train shape:", train.shape)
print("Test shape: ", test.shape)

Train shape: (630000, 21)
Test shape:  (270000, 20)


In [3]:
# Separate features and target
TARGET = 'Irrigation_Need'
DROP_COLS = ['id', TARGET]

X = train.drop(columns=DROP_COLS)
y = train[TARGET]
X_test = test.drop(columns=['id'])

# Handle any categorical columns
X = pd.get_dummies(X)
X_test = pd.get_dummies(X_test)
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

# XGBoost needs the target as numbers not text
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Features shape:", X.shape)
print("Classes:", le.classes_)
print("Encoded as:", list(range(len(le.classes_))))

Features shape: (630000, 43)
Classes: ['High' 'Low' 'Medium']
Encoded as: [0, 1, 2]


In [4]:
# Train XGBoost
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    objective='multi:softmax',
    num_class=3,
    eval_metric='mlogloss',
    n_jobs=-1
)

# Cross validation
cv_scores = cross_val_score(xgb, X, y_encoded, cv=5, scoring='balanced_accuracy')

print(f"XGBoost CV Balanced Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Individual fold scores: {cv_scores.round(4)}")

XGBoost CV Balanced Accuracy: 0.9620 ± 0.0020
Individual fold scores: [0.9628 0.9622 0.9641 0.9626 0.9582]


In [5]:
# Fit the model on full training data
xgb.fit(X, y_encoded)

# Make predictions on test set
preds_encoded = xgb.predict(X_test)

# Convert numbers back to Low/Medium/High
preds = le.inverse_transform(preds_encoded)

# Save submission file
submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': preds
})
submission.to_csv('submission.csv', index=False)

print("Submission saved!")
print("Prediction distribution:")
print(pd.Series(preds).value_counts())

Submission saved!
Prediction distribution:
Low       159864
Medium    101475
High        8661
Name: count, dtype: int64
